# 09 — Text-to-Speech (TTS)

Dans ce notebook, nous allons construire un pipeline simple de synthèse vocale.

Pipeline général :

```text
Text
→ Text normalization / phonemes
→ TTS model
→ Audio waveform

Objectifs :

```text

- comprendre le pipeline TTS ;
- convertir du texte en parole ;
- utiliser un modèle TTS préentraîné ;
- générer et sauvegarder un fichier audio ;
- écouter et visualiser le waveform ;
- tester plusieurs phrases ;
- comparer différentes voix si le modèle le permet ;
- vérifier le sample rate et la durée ;
- préparer la suite vers le voice cloning.

In [1]:
!python -m pip install -U "kokoro==0.7.16" soundfile

You should consider upgrading via the '/Users/ziyadamine/Documents/weather-audio-learning/.venv/bin/python -m pip install --upgrade pip' command.


In [ ]:
import kokoro
from kokoro import KPipeline

print("Kokoro version :", kokoro.__version__)
print("KPipeline chargé avec succès.")

/Users/ziyadamine/Documents/weather-audio-learning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
from kokoro import KPipeline

pipeline = KPipeline(lang_code="a")

text = "Hi, I'm Iyad, AI and ML engineer in progress"

generator = pipeline(
    text,
    voice="af_heart",
    speed=1.0
)

In [ ]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

In [ ]:
import soundfile as sf

for i, (graphemes, phonemes, audio) in enumerate(generator):
    sf.write(
        f"outputs/kokoro_tts_{i}.wav",
        audio.cpu().numpy(),
        24000
    )
  
    print(f"Saved: outputs/kokoro_tts_{i}.wav")

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        "outputs/kokoro_tts_0.wav"
    )
)

In [ ]:
import soundfile as sf

audio_data, sample_rate = sf.read(
    "outputs/kokoro_tts_0.wav"
)

duration = len(audio_data) / sample_rate

print("Sample rate :", sample_rate)
print("Number of samples :", len(audio_data))
print("Duration :", duration)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

time_axis = np.arange(len(audio_data)) / sample_rate

plt.figure(figsize=(10, 4))
plt.plot(time_axis, audio_data)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Kokoro TTS Waveform")
plt.grid(True)

plt.show()

In [ ]:
test_texts = [
    "The temperature is 25 degrees.",
    "The meeting is on October 5, 2026.",
    "Dr. Smith works at 10 a.m.",
    "AI and ML are changing the world."
]

for text in test_texts:
    print("\nTEXT:", text)

    generator = pipeline(
        text,
        voice="af_heart",
        speed=1.0
    )

    for graphemes, phonemes, audio in generator:
        print("Phonemes:", phonemes)

## Text Normalization and Phonemization

Avant de générer l'audio, Kokoro transforme le texte en une représentation phonétique.

Quelques observations :

- `25` est interprété comme `twenty five` ;
- `October 5, 2026` est converti en une prononciation naturelle de la date ;
- `Dr.` est développé en `doctor` ;
- `10 a.m.` est interprété comme `ten A M` ;
- `AI` est prononcé lettre par lettre ;
- `ML` est également prononcé comme un acronyme.

Cette étape est importante car le modèle TTS ne dépend pas uniquement de l'orthographe brute du texte.

Le frontend transforme le texte en une représentation plus proche de la prononciation réelle :

```text
Raw text
→ normalization
→ phonemes
→ speech generation

In [ ]:
speeds = [0.8, 1.0, 1.2]

text = "Hi, I'm Iyad, AI and ML engineer in progress."

for speed in speeds:
    generator = pipeline(
        text,
        voice="af_heart",
        speed=speed
    )

    for i, (graphemes, phonemes, audio) in enumerate(generator):
        duration = len(audio) / 24000

        print(
            f"Speed = {speed} | "
            f"Samples = {len(audio)} | "
            f"Duration = {duration:.2f} s"
        )

## Conclusion

Dans ce notebook, nous avons construit un pipeline simple de Text-to-Speech avec Kokoro.

Nous avons appris à :

- charger un modèle TTS préentraîné ;
- convertir du texte en parole ;
- observer les phonèmes générés par le text frontend ;
- sauvegarder la sortie audio en fichier WAV ;
- écouter l'audio directement dans Jupyter ;
- vérifier le sample rate et la durée ;
- visualiser le waveform ;
- comprendre la normalisation des nombres, dates, abréviations et acronymes ;
- observer l'effet du paramètre `speed` sur la durée de la parole.

### Résultat principal

Pipeline utilisé :

```text
Text
→ normalization
→ phonemes
→ Kokoro TTS
→ waveform
→ WAV